# Exploratory Data Analysis — Company Dataset

In [1]:
import json
import ast
import pandas as pd
import statistics 
from collections import Counter
from collections import defaultdict

In [2]:
with open('companies.jsonl') as file:
    companies = [json.loads(line) for line in file if line.strip()]

number_companies = len(companies)
print(f'Total companies in file: {number_companies}')

Total companies in file: 477


### 1. Field coverage

In [3]:
def get_adress(company):
    adress = company.get('address')
    if isinstance(adress, dict):
        return adress
    if isinstance(adress, str):
        try:
            parsed = ast.literal_eval(adress)
            if isinstance(parsed, dict):
                return parsed
        except:
            pass
    return {}

ADDR_KEYS = [ 'country_code', 'country_name', 'county', 'house_number', 'latitude', 'longitude', 'postcode', 'raw_text',
    'region_code', 'region_name', 'street', 'suburb', 'town' ]

fields = {
    'operational_name': lambda company: bool(company.get('operational_name')),
    'website': lambda company: bool(company.get('website')),
    'description': lambda company: bool(company.get('description')),
    'primary_naics': lambda company: bool(company.get('primary_naics')),
    'secondary_naics': lambda company: bool(company.get('secondary_naics')),
    'business_model': lambda company: bool(company.get('business_model') and len(company['business_model']) > 0),
    'core_offerings': lambda company: bool(company.get('core_offerings') and len(company['core_offerings']) > 0),
    'target_markets': lambda company: bool(company.get('target_markets') and len(company['target_markets']) > 0),
    'is_public': lambda company: company.get('is_public') is not None,
    'employee_count': lambda company: company.get('employee_count') is not None,
    'revenue': lambda company: company.get('revenue') is not None,
    'year_founded': lambda company: company.get('year_founded') is not None,
}

# Add all 13 address sub-fields
for key in ADDR_KEYS:
    fields[f'addr.{key}'] = lambda company, k=key: bool(get_adress(company).get(k))


print(f'{"Field":<22} {"Present":>7} {"Missing":>7} {"Coverage":>9}')
print('=' * 50)

for field, check in fields.items():
    present = sum(1 for company in companies if check(company))
    missing = number_companies - present
    pct = present / number_companies * 100
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f'{field:<22} {present:>7} {missing:>7} {pct:>7.1f}%  {bar}')

Field                  Present Missing  Coverage
operational_name           475       2    99.6%  ███████████████████░
website                    439      38    92.0%  ██████████████████░░
description                477       0   100.0%  ████████████████████
primary_naics              477       0   100.0%  ████████████████████
secondary_naics             11     466     2.3%  ░░░░░░░░░░░░░░░░░░░░
business_model             477       0   100.0%  ████████████████████
core_offerings             477       0   100.0%  ████████████████████
target_markets             477       0   100.0%  ████████████████████
is_public                  477       0   100.0%  ████████████████████
employee_count             289     188    60.6%  ████████████░░░░░░░░
revenue                    384      93    80.5%  ████████████████░░░░
year_founded               346     131    72.5%  ██████████████░░░░░░
addr.country_code          477       0   100.0%  ████████████████████
addr.country_name           20     457   

In [13]:
group_by_count = defaultdict(list)
group_fields = defaultdict(lambda: Counter())

for c in companies:
    addr = get_adress(c)
    present_keys = [k for k in ADDR_KEYS if bool(addr.get(k))]
    n = len(present_keys)
    group_by_count[n].append(c.get('operational_name', '??'))
    for k in present_keys:
        group_fields[n][k] += 1

print(f'{"# fields":>9} {"# companies":>12}  {"Fields present"}')
print('=' * 80)
for n in sorted(group_by_count.keys(), reverse=True):
    names = group_by_count[n]
    field_summary = ', '.join(
        f'{k}({v})' for k, v in sorted(group_fields[n].items(), key=lambda x: -x[1])
    )
    print(f'{n:>9} {len(names):>12}  {field_summary}')

print(f'\n{"Total":>9} {sum(len(v) for v in group_by_count.values()):>12}')



 # fields  # companies  Fields present
       13            2  country_code(2), country_name(2), county(2), house_number(2), latitude(2), longitude(2), postcode(2), raw_text(2), region_code(2), region_name(2), street(2), suburb(2), town(2)
       12            7  country_code(7), country_name(7), house_number(7), latitude(7), longitude(7), postcode(7), raw_text(7), region_code(7), region_name(7), street(7), town(7), suburb(6), county(1)
       11            8  country_code(8), country_name(8), house_number(8), latitude(8), longitude(8), postcode(8), raw_text(8), region_code(8), region_name(8), street(8), town(8)
        9            3  country_code(3), country_name(3), latitude(3), longitude(3), postcode(3), raw_text(3), region_code(3), region_name(3), town(3)
        5          437  country_code(437), latitude(437), longitude(437), region_name(437), town(437)
        4            4  country_code(4), latitude(4), longitude(4), region_name(4)
        3           16  country_code(16), la

!! **Observation**

> The address data is highly sparse: only **2 out of 477** companies have all 13 address sub-fields populated, and the overwhelming majority (437 companies, ~92%) have only 5 fields available. The fields with near-universal coverage are:
> - `country_code` — 100% (477/477)
> - `latitude` / `longitude` — 99.4% (474/477)
> - `region_name` — 97.3% (464/477)
> - `town` — 96.4% (460/477)
>
> All remaining fields (`street`, `house_number`, `postcode`, `county`, `suburb`, `raw_text`, `region_code`, `country_name`) fall below 5% coverage - they are present for at most 20 companies.
>
> Given this distribution, the qualification pipeline uses **only `country_code`** (for location-based hard filters), **`latitude`/`longitude`** (for geo-proximity scoring), and **`region_name`/`town`** (for text-based location matching). Attempting deeper address matching (e.g. street-level or postal code filtering) would be unreliable with data available for fewer than 4% of records, and would not provide any visible benefit for the qualification task.


In [16]:
# Show companies in the top groups
for n in sorted(group_by_count.keys(), reverse=True):
    names = group_by_count[n]
    if n >= 8:
        print(f'\n--- {n} fields ({len(names)} companies) ---')
        for name in names:
            print(f'  - {name}')


--- 13 fields (2 companies) ---
  - Ariv Pay
  - VIP Exchange

--- 12 fields (7 companies) ---
  - Aircash
  - FGC Money Transfer
  - ECrowd Invest
  - New Payment Innovation
  - Phin
  - Credito21
  - Rantum Capital

--- 11 fields (8 companies) ---
  - Mangal Money Transfer
  - EasyKassa
  - City Exchange
  - Brink's Solutions Nederland
  - Salauddin Global Trading
  - CTC FUND
  - Folio
  - Agility FX

--- 9 fields (3 companies) ---
  - SkyHarvest
  - Euro-Rijn Financial Services
  - European Pay


In [ ]:
COLUMNS = [
    'operational_name', 'website', 'year_founded', 'address', 'employee_count', 'revenue', 'primary_naics',
    'secondary_naics', 'description', 'business_model', 'core_offerings', 'target_markets', 'is_public' ]

rows = []
for c in companies:
    row = {}
    for col in COLUMNS:
        val = c.get(col)
        if val is None:
            row[col] = None
        elif isinstance(val, (dict, list)):
            row[col] = json.dumps(val, ensure_ascii=False)
        else:
            row[col] = val
    rows.append(row)

data = pd.DataFrame(rows, columns=COLUMNS)

In [6]:
type_rows = []
for copmany in companies:
    row = {}
    for col in COLUMNS:
        value = copmany.get(col)
        row[col] = type(value).__name__
    type_rows.append(row)

types_df = pd.DataFrame(type_rows, columns=COLUMNS)

print('Value types per column:')
print('=' * 50)
for col in COLUMNS:
    counts = types_df[col].value_counts().to_dict()
    print(f'{col}:')
    for type, count in counts.items():
        print(f'  {type:>10s}: {count:>4d}')
    print()

Value types per column:
operational_name:
         str:  475
    NoneType:    2

website:
         str:  439
    NoneType:   38

year_founded:
       float:  346
    NoneType:  131

address:
        dict:  413
         str:   64

employee_count:
       float:  289
    NoneType:  188

revenue:
       float:  384
    NoneType:   93

primary_naics:
        dict:  413
         str:   64

secondary_naics:
    NoneType:  466
        dict:   11

description:
         str:  477

business_model:
        list:  477

core_offerings:
        list:  477

target_markets:
        list:  477

is_public:
        bool:  477



!! **Observation - Format inconsistencies**:

> The type analysis reveals that `address` and `primary_naics` each have **two different representations**: 413 records store them as proper `dict` objects, while 64 records store them as `str` (serialized dictionaries). This is a format inconsistency in the source data - the same 64 companies are affected in both fields (confirmed below). These string-encoded records need to be parsed back into dictionaries before they can be used reliably in the pipeline.                                                                                            

> Additionally, several numeric fields have significant `NoneType` counts due to missing values: `employee_count` (188 missing, ~39%), `year_founded` (131 missing, ~27%), and `revenue` (93 missing, ~20%). This limits the reliability of hard filters on these fields — a strict filter on employee count , for example, would silently exclude nearly 40% of the dataset. 

In [7]:
# Show companies where address is a string instead of dict
string_mask = [isinstance(c.get('address'), str) for c in companies]
print(f'Address as string: {sum(string_mask)} companies')
print()
data[string_mask][['operational_name', 'address']].head(10)

Address as string: 64 companies



,operational_name,address
0,Rompetrol,"{'country_code': 'ro', 'latitude': 44.4775537,..."
1,Unilever,"{'country_code': 'ro', 'latitude': 44.4761769,..."
2,Rompetrol,"{'country_code': 'ro', 'latitude': 44.47775356..."
3,CBRE Romania,"{'country_code': 'ro', 'latitude': 44.4540414,..."
4,TIMAC AGRO,"{'country_code': 'ro', 'latitude': 44.4792186,..."
5,OMV,"{'country_code': 'ro', 'latitude': 44.490331, ..."
6,Compania Națională de Căi Ferate CFR,"{'country_code': 'ro', 'latitude': 44.4493708,..."
7,Mercedes-Benz Trucks & Buses Romania,"{'country_code': 'ro', 'latitude': 44.49860626..."
8,Dacia,"{'country_code': 'ro', 'latitude': 44.429628, ..."
9,Fildas Trading,"{'country_code': 'ro', 'latitude': 44.4781612,..."


In [8]:
string_naics = [isinstance(c.get('primary_naics'), str) for c in companies]
print(f'NAICS as string: {sum(string_naics)} companies')
print()
data[string_naics][['operational_name', 'primary_naics']].head(10)

NAICS as string: 64 companies



,operational_name,primary_naics
0,Rompetrol,"{'code': '324110', 'label': 'Petroleum Refiner..."
1,Unilever,"{'code': '424690', 'label': 'Other Chemical an..."
2,Rompetrol,"{'code': '324110', 'label': 'Petroleum Refiner..."
3,CBRE Romania,"{'code': '531210', 'label': 'Offices of Real E..."
4,TIMAC AGRO,"{'code': '325311', 'label': 'Nitrogenous Ferti..."
5,OMV,"{'code': '457120', 'label': 'Other Gasoline St..."
6,Compania Națională de Căi Ferate CFR,"{'code': '488210', 'label': 'Support Activitie..."
7,Mercedes-Benz Trucks & Buses Romania,"{'code': '441110', 'label': 'New Car Dealers'}"
8,Dacia,"{'code': '336110', 'label': 'Automobile and Li..."
9,Fildas Trading,"{'code': '424210', 'label': ""Drugs and Druggis..."


In [9]:
both = [a and n for a, n in zip(string_mask, string_naics)]
only_addr = [a and not n for a, n in zip(string_mask, string_naics)]
only_naics = [not a and n for a, n in zip(string_mask, string_naics)]

print(f'Both address + NAICS as string: {sum(both)}')
print(f'Only address as string:         {sum(only_addr)}')
print(f'Only NAICS as string:           {sum(only_naics)}')
print()
print('→ Same batch of companies, same serialization issue.')

Both address + NAICS as string: 64
Only address as string:         0
Only NAICS as string:           0

→ Same batch of companies, same serialization issue.


In [10]:
missing = data.isnull().sum()
total = len(data)

print(f'{"Column":<20} {"Present":>7} {"Missing":>7} {"Coverage":>9}')
print('-' * 48)
for col in COLUMNS:
    present = total - missing[col]
    pct = present / total * 100
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f'{col:<20} {present:>7} {missing[col]:>7} {pct:>7.1f}%  {bar}')

Column               Present Missing  Coverage
------------------------------------------------
operational_name         475       2    99.6%  ███████████████████░
website                  439      38    92.0%  ██████████████████░░
year_founded             346     131    72.5%  ██████████████░░░░░░
address                  477       0   100.0%  ████████████████████
employee_count           289     188    60.6%  ████████████░░░░░░░░
revenue                  384      93    80.5%  ████████████████░░░░
primary_naics            477       0   100.0%  ████████████████████
secondary_naics           11     466     2.3%  ░░░░░░░░░░░░░░░░░░░░
description              477       0   100.0%  ████████████████████
business_model           477       0   100.0%  ████████████████████
core_offerings           477       0   100.0%  ████████████████████
target_markets           477       0   100.0%  ████████████████████
is_public                477       0   100.0%  ████████████████████


### 2. Revenue and employee distributions

In [ ]:
revenues = [c['revenue'] for c in companies if c.get('revenue') is not None]
print(f"Revenue: {len(revenues)}/{number_companies} companies ({len(revenues)/number_companies*100:.0f}%)")
print(f"  Min: ${min(revenues):,.0f}  |  Median: ${statistics.median(revenues):,.0f}  |  Max: ${max(revenues):,.0f}")

revenue_brackets = [
    (0,       1e6,          "<$1M"),
    (1e6,     10e6,        "$1–10M"),
    (10e6,    100e6,      "$10–100M"),
    (100e6,   1e9,       "$100M–1B"),
    (1e9,     float('inf'), ">$1B"),
]

for lo, hi, label in revenue_brackets:
    count = sum(1 for r in revenues if lo <= r < hi)
    print(f"  {label:>10s}: {count:>4d}  {'#' * max(1, count // 3)}")


employees = [c['employee_count'] for c in companies if c.get('employee_count') is not None]
print(f"\nEmployees: {len(employees)}/{number_companies} companies ({len(employees)/number_companies*100:.0f}%)")
print(f"  Min: {min(employees):,.0f}  |  Median: {statistics.median(employees):,.0f}  |  Max: {max(employees):,.0f}")

employee_brackets = [
    (0,       10,           "<10"),
    (10,      50,          "10–50"),
    (50,      200,        "50–200"),
    (200,     1000,      "200–1K"),
    (1000,    10000,     "1K–10K"),
    (10000,   float('inf'), ">10K"),
]

for lo, hi, label in employee_brackets:
    count = sum(1 for e in employees if lo <= e < hi)
    print(f"  {label:>8s}: {count:>4d}  {'#' * max(1, count // 3)}")

Revenue: 384/477 companies (81%)
  Min: $7,524  |  Median: $20,184,482  |  Max: $2,500,388,591,944
        <$1M:   71  #######################
      $1–10M:   82  ###########################
    $10–100M:   81  ###########################
    $100M–1B:   39  #############
        >$1B:  111  #####################################

Employees: 289/477 companies (61%)
  Min: 1  |  Median: 30  |  Max: 2,100,000
       <10:  100  #################################
     10–50:   56  ##################
    50–200:   18  ######
    200–1K:   31  ##########
    1K–10K:   27  #########
      >10K:   57  ###################


### 3. Description details


In [21]:
desc_lengths = [len(c.get('description', '')) for c in companies if c.get('description')]                                                                                           
print(f"Description: min {min(desc_lengths)}, median {statistics.median(desc_lengths):.0f}, max {max(desc_lengths)} chars")                                                        

# --- Text blob size (combined input for BM25 + embeddings) ---                                                                                                                     
def build_text_blob(company):                                              
    """Concatenate all text fields that feed into the scoring pipeline."""                                                                                                                
    description   = company.get('description', '')                                                                                                                                        
    offerings     = company.get('core_offerings') or []                                                                                                                                   
    markets       = company.get('target_markets') or []                                                                                                                             
    business_model = company.get('business_model') or []
    return ' '.join([description] + offerings + markets + business_model)

blob_lengths = [len(build_text_blob(c)) for c in companies]

print(f"Text blob (BM25 + embeddings input): min {min(blob_lengths)}, median {statistics.median(blob_lengths):.0f}, max {max(blob_lengths)} chars")

Description: min 121, median 558, max 913 chars
Text blob (BM25 + embeddings input): min 192, median 1126, max 1812 chars


**Observation - Text coverage for scoring**:

> Every company has a non-empty description (100% coverage), and all 477 records include `business_model`, `core_offerings`, and `target_markets` lists. The combined text blob - which serves as input for both BM25 keyword scoring and semantic embeddings. Although the documents are short, BM25 normalizes for document length and the text contains specific industry terms ("petroleum refining", "SaaS") that are enough to distinguish companies from each other effectively.